# Evo 2 generation for PRISM / ICBINB — **run me on an A100 High-RAM Colab runtime**

**What this is.** A single, self-contained notebook that generates Evo 2 (single-nucleotide genomic
LM) continuations for 26 gene prompts. Its output is the drop-in *third model* the ICBINB paper needs
(the single-nucleotide control alongside Carbon and GENERator). **You do not need any other files** —
the prompts are embedded below.

**Runtime (important).** `Runtime → Change runtime type → A100 GPU` **and** `High-RAM`. Evo 2 (7B) OOMs
on T4/L4 free tiers. The one-time weight download is ~14 GB — **do not interrupt it**.

**What to send back.** When it finishes, the last cell makes `evo2_output.zip`. Download it and send
it back — that's the whole deliverable. Total run time is typically ~15–40 min on an A100.


### 1. Confirm you actually have an A100 (this step is cheap and saves a wasted hour)


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)
# You want to see an A100 with ~40960 MiB. A T4/L4/V100 will OOM on the 7B model.


### 2. Install Evo 2 (one-time)

⚠️ **This is the step most likely to need patience or a small tweak.** `flash-attn` sometimes
compiles from source on Colab (can take 10–20 min) instead of using a prebuilt wheel. If it errors,
re-run the cell once; if it still errors, send the PRISM author the error text — it's usually a
one-line version pin. The Evo 2 weights are public on Hugging Face (no token or license click
needed); they download automatically on first model load.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # set BEFORE torch import
!pip install -q flash-attn==2.8.0.post2 --no-build-isolation
!pip install -q evo2
import evo2, torch
print("install done | torch", torch.__version__, "| cuda", torch.cuda.is_available())


### 3. Write the embedded prompts (no upload needed — 26 gene CDS prompts are baked in)


In [ ]:
import base64
_PROMPTS_B64 = (
    "..."
)
with open("prompts.fasta", "wb") as f:
    f.write(base64.b64decode(_PROMPTS_B64))
from Bio import SeqIO
n = sum(1 for _ in SeqIO.parse("prompts.fasta", "fasta"))
print(f"wrote prompts.fasta with {n} sequences")


### 3.5 Smoke test (~2–3 min) — **do this before the full run**

Loads the model and generates ONE short continuation. If this cell works, the full run in step 4 is
just the same call in a loop. If it errors, the fix is quick and nothing has been wasted on a long run.
The only thing that ever needs adjusting is the `generate(...)` argument names, if Arc changes the API.


In [ ]:
import torch
from evo2 import Evo2
print("Loading evo2_7b (~14GB first time; DO NOT interrupt the download)...")
model = Evo2("evo2_7b")
print("loaded. running a 1-sequence smoke test...")
with torch.inference_mode():
    out = model.generate(prompt_seqs=["ATGGCGACGACGTTAGCTAGCGATCGATCGTAGCTAGCTAGCAT"],
                         n_tokens=32, temperature=1.0, top_k=4, verbose=0)
seqs = out.sequences if hasattr(out, "sequences") else out[0]
print("OK -- sample continuation:", seqs[0][:60])
print("If you see a DNA string above, everything works. Proceed to step 4.")


### 4. Generate (RAM-frugal loop: small batch, per-gene write, VRAM cleared each batch)

Idempotent — if the runtime disconnects, just re-run this cell and it skips genes already done.


In [ ]:
import re, gc, torch, os
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from evo2 import Evo2

PROMPTS = "prompts.fasta"
OUT = "evo2_output"; os.makedirs(OUT, exist_ok=True)

# knobs: N continuations/gene, batch size, max new tokens, prefix fraction, sampling
N, BATCH, MAXTOK, PREFIX, TEMP, TOPK = 30, 2, 400, 0.2, 1.0, 4

if "model" not in globals():   # reuse the model from the smoke test if it's still loaded
    print("Loading evo2_7b (~14GB, one-time; bf16 fits an A100). DO NOT interrupt...")
    model = Evo2("evo2_7b")
print("model ready.")
clean = lambda s: re.sub(r"[^ACGTN]", "", re.sub(r"<[^>]+>", "", s.upper()))

for rec in SeqIO.parse(PROMPTS, "fasta"):
    gid, seq = rec.id, str(rec.seq).upper().replace(" ", "")
    cut = int(len(seq) * PREFIX); prefix, suffix = seq[:cut], seq[cut:]
    safe = gid.replace("/", "_").replace(" ", "_").replace(":", "_")
    outp = os.path.join(OUT, f"{safe}_generated.fasta")
    if not prefix or os.path.exists(outp):
        print("skip", gid); continue
    recs, n = [], 0
    while n < N:
        b = min(BATCH, N - n)
        with torch.inference_mode():
            out = model.generate(prompt_seqs=[prefix]*b, n_tokens=MAXTOK,
                                 temperature=TEMP, top_k=TOPK, verbose=0)
        seqs = out.sequences if hasattr(out, "sequences") else out[0]
        for s in seqs:
            cont = s[len(prefix):] if s.upper().startswith(prefix.upper()) else s
            recs.append(SeqRecord(Seq(clean(cont)), id=f"{safe}|sample_{n+1:03d}", description="evo2"))
            n += 1
        del out, seqs; gc.collect(); torch.cuda.empty_cache()
    SeqIO.write(recs, outp, "fasta")
    if suffix:
        SeqIO.write([SeqRecord(Seq(suffix), id=safe, description="true continuation")],
                    os.path.join(OUT, f"{safe}_expected_continuation.fasta"), "fasta")
    print(f"{gid}: wrote {len(recs)} continuations")
    del recs; gc.collect(); torch.cuda.empty_cache()
print("DONE ->", OUT)


### 5. Zip the output and download — **this file is the deliverable**


In [ ]:
import shutil
shutil.make_archive("evo2_output", "zip", "evo2_output")
print("created evo2_output.zip")
try:
    from google.colab import files
    files.download("evo2_output.zip")
except Exception as e:
    print("auto-download unavailable; grab evo2_output.zip from the file browser on the left.", e)


---
### For the PRISM author (runs locally on CPU, after you get `evo2_output.zip` back)
Unzip into the repo, then the Evo 2 half of ICBINB is one command each — no GPU needed:
```bash
python stress_test_generation.py \
    --model_dir carbon:results/genmodel_bias/carbon_output \
    --model_dir generator:results/genmodel_bias/generator_output \
    --model_dir evo2:evo2_output \
    --reference_fasta prompts.fasta --prefix_frac 0.2 \
    --cosmic_dir <COSMIC_PROFILE_dir> --output_dir results/stress_test
python fill_paper_numbers.py --stress_dir results/stress_test \
    --sweep_dir results/sweep_analysis --paper_dir <icbinb tex dir>
```
This fills ICBINB's three pending Evo 2 markers (the single-nucleotide third violin). The stress-test
is spectrum/k-mer based, so it runs fine on the `python3.10` CPU environment.
